# Incident Response Runbook: Resource Development - Acquire Infrastructure

**Tactic:** Resource Development
**Technique:** Acquire Infrastructure (T1583)
**Severity:** LOW

## Overview

This runbook provides step-by-step incident response procedures for Acquire Infrastructure activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add project root to path for imports
sys.path.append(os.path.join(os.path.dirname(__file__), '..', '..'))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()
affected_systems = []
incident_id = None

# Query Splunk for infrastructure acquisition indicators
print(f"\n[DETECTION] Querying Splunk for infrastructure acquisition activities...")
try:
    # Domain registration anomalies
    domain_query = """
    index=* sourcetype=dns OR sourcetype=web_proxy
    | search (domain_registrar OR whois OR domain_purchase)
    | eval risk_score = case(
        match(domain, "\.(tk|ml|ga|cf)$"), 8,
        match(domain, "\d{4,}"), 7,
        match(domain, "(test|demo|staging)"), 6,
        match(domain, ".*-.*-.*-.*"), 9,
        1==1, 3
    )
    | where risk_score >= 6
    | stats count by domain, src_ip, user, risk_score
    | sort -risk_score
    """

    domain_results = splunk.search(domain_query, timeframe="-24h")
    if domain_results:
        print(f"   Found {len(domain_results)} suspicious domain registrations")
        for result in domain_results[:5]:  # Show top 5
            affected_systems.append({
                'type': 'domain_registration',
                'domain': result.get('domain'),
                'src_ip': result.get('src_ip'),
                'user': result.get('user'),
                'risk_score': result.get('risk_score')
            })

    # Cloud resource provisioning anomalies
    cloud_query = """
    index=* sourcetype=aws_cloudtrail OR sourcetype=azure_activity OR sourcetype=gcp_audit
    | search (CreateBucket OR CreateInstance OR CreateVM OR AllocatePublicIP OR CreateDomain)
    | eval risk_score = case(
        match(resource_name, "\d{8,}"), 8,
        match(resource_name, "(test|temp|staging)"), 5,
        match(resource_name, ".*random.*"), 7,
        match(location, "(Russia|China|North Korea)"), 9,
        1==1, 4
    )
    | where risk_score >= 6
    | stats count by resource_name, user, location, service, risk_score
    | sort -risk_score
    """

    cloud_results = splunk.search(cloud_query, timeframe="-24h")
    if cloud_results:
        print(f"   Found {len(cloud_results)} suspicious cloud resource provisions")
        for result in cloud_results[:5]:  # Show top 5
            affected_systems.append({
                'type': 'cloud_provisioning',
                'resource_name': result.get('resource_name'),
                'user': result.get('user'),
                'location': result.get('location'),
                'service': result.get('service'),
                'risk_score': result.get('risk_score')
            })

    # IP address acquisition patterns
    ip_query = """
    index=* sourcetype=firewall OR sourcetype=netflow
    | search (ip_allocation OR ip_purchase OR ip_lease)
    | eval risk_score = case(
        match(ip_range, "/24|/16|/8"), 9,
        match(ip_range, "/32"), 3,
        match(provider, "(bulletproof|anonymous)"), 10,
        1==1, 5
    )
    | where risk_score >= 7
    | stats count by ip_range, provider, user, risk_score
    | sort -risk_score
    """

    ip_results = splunk.search(ip_query, timeframe="-24h")
    if ip_results:
        print(f"   Found {len(ip_results)} suspicious IP acquisitions")
        for result in ip_results[:5]:  # Show top 5
            affected_systems.append({
                'type': 'ip_acquisition',
                'ip_range': result.get('ip_range'),
                'provider': result.get('provider'),
                'user': result.get('user'),
                'risk_score': result.get('risk_score')
            })

except Exception as e:
    print(f"   Splunk query failed: {e}")

# Check CrowdStrike for related endpoint activity
print(f"\n[DETECTION] Checking CrowdStrike for related endpoint activity...")
try:
    for system in affected_systems:
        if system.get('src_ip'):
            cs_alerts = crowdstrike.get_alerts_by_ip(system['src_ip'])
            if cs_alerts:
                system['cs_alerts'] = len(cs_alerts)
                print(f"   Found {len(cs_alerts)} CrowdStrike alerts for IP {system['src_ip']}")
except Exception as e:
    print(f"   CrowdStrike check failed: {e}")

# Enrich with threat intelligence
print(f"\n[INTEL] Enriching with threat intelligence...")
try:
    for system in affected_systems:
        if system.get('domain'):
            ti_data = misp.lookup_domain(system['domain'])
            if ti_data:
                system['threat_intel'] = ti_data
                print(f"   Threat intel found for domain: {system['domain']}")
        elif system.get('ip_range'):
            ti_data = misp.lookup_ip(system['ip_range'].split('/')[0])
            if ti_data:
                system['threat_intel'] = ti_data
                print(f"   Threat intel found for IP: {system['ip_range']}")
except Exception as e:
    print(f"   Threat intelligence lookup failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    case_data = {
        'title': f'Infrastructure Acquisition Activity - {len(affected_systems)} indicators',
        'description': f'Detected suspicious infrastructure acquisition activities across {len(affected_systems)} indicators',
        'severity': 'LOW',
        'tactic': 'Resource Development',
        'technique': 'Acquire Infrastructure (T1583)',
        'indicators': affected_systems,
        'detection_time': detection_time
    }
    incident_id = iris.create_case(case_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")

print(f"\n Detection complete:")
print(f"  - {len(affected_systems)} infrastructure acquisition indicators detected")
print(f"  - Threat intelligence enriched for {len([s for s in affected_systems if s.get('threat_intel')])} indicators")
print(f"  - IRIS case created: {incident_id}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
isolated_systems = []
blocked_domains = []
blocked_ips = []
disabled_accounts = []

# Isolate affected systems
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        if system.get('src_ip'):
            # Find device by IP and isolate
            devices = crowdstrike.get_devices_by_ip(system['src_ip'])
            for device in devices:
                isolation_result = crowdstrike.isolate_host(device['device_id'])
                if isolation_result.get('status') == 'isolated':
                    isolated_systems.append(device.get('hostname', system['src_ip']))
                    print(f"   Isolated system: {device.get('hostname', system['src_ip'])}")
    except Exception as e:
        print(f"   System isolation failed for {system.get('src_ip', 'unknown')}: {e}")

# Block suspicious domains
print(f"\n[ACTION] Blocking suspicious domains...")
for system in affected_systems:
    try:
        if system.get('domain'):
            block_result = shuffle.block_domain(system['domain'])
            if block_result.get('status') == 'blocked':
                blocked_domains.append(system['domain'])
                print(f"   Blocked domain: {system['domain']}")
    except Exception as e:
        print(f"   Domain blocking failed for {system.get('domain')}: {e}")

# Block suspicious IP ranges
print(f"\n[ACTION] Blocking suspicious IP ranges...")
for system in affected_systems:
    try:
        if system.get('ip_range'):
            block_result = shuffle.block_ip_range(system['ip_range'])
            if block_result.get('status') == 'blocked':
                blocked_ips.append(system['ip_range'])
                print(f"   Blocked IP range: {system['ip_range']}")
    except Exception as e:
        print(f"   IP blocking failed for {system.get('ip_range')}: {e}")

# Disable compromised accounts
print(f"\n[ACTION] Disabling compromised accounts...")
unique_users = set()
for system in affected_systems:
    if system.get('user') and system['user'] != 'unknown':
        unique_users.add(system['user'])

for user in unique_users:
    try:
        disable_result = shuffle.disable_user_account(user)
        if disable_result.get('status') == 'disabled':
            disabled_accounts.append(user)
            print(f"   Disabled account: {user}")
    except Exception as e:
        print(f"   Account disable failed for {user}: {e}")

# Enable enhanced monitoring
print(f"\n[ACTION] Enabling enhanced monitoring...")
monitoring_rules = []
try:
    # Enable CrowdStrike infrastructure monitoring
    cs_monitoring = crowdstrike.enable_enhanced_monitoring('infrastructure_acquisition')
    if cs_monitoring.get('status') == 'enabled':
        monitoring_rules.append('CrowdStrike infrastructure monitoring')
        print(f"   Enabled CrowdStrike infrastructure acquisition monitoring")

    # Enable Splunk infrastructure correlation rules
    splunk_monitoring = splunk.enable_correlation_rule('infrastructure_acquisition')
    if splunk_monitoring.get('status') == 'enabled':
        monitoring_rules.append('Splunk infrastructure correlation')
        print(f"   Enabled Splunk infrastructure acquisition correlation rules")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

# Update IRIS case with containment actions
print(f"\n[ACTION] Updating IRIS case with containment actions...")
try:
    containment_summary = {
        'isolated_systems': len(isolated_systems),
        'blocked_domains': len(blocked_domains),
        'blocked_ips': len(blocked_ips),
        'disabled_accounts': len(disabled_accounts),
        'monitoring_enabled': len(monitoring_rules),
        'containment_time': containment_time
    }
    iris.update_case(incident_id, 'containment_complete', containment_summary)
    print(f"   Updated IRIS case {incident_id} with containment results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_systems)} systems isolated")
print(f"  - {len(blocked_domains)} domains blocked")
print(f"  - {len(blocked_ips)} IP ranges blocked")
print(f"  - {len(disabled_accounts)} accounts disabled")
print(f"  - {len(monitoring_rules)} enhanced monitoring rules enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
terminated_processes = []
removed_resources = []
cleaned_artifacts = []

# Terminate infrastructure acquisition processes
print(f"\n[ACTION] Terminating infrastructure acquisition processes...")
for system in affected_systems:
    try:
        if system.get('src_ip'):
            devices = crowdstrike.get_devices_by_ip(system['src_ip'])
            for device in devices:
                processes = crowdstrike.get_suspicious_processes(device['device_id'], 'infrastructure_acquisition')
                for proc in processes:
                    if proc.get('process_name') in ['domain_registrar.exe', 'cloud_provisioner.exe', 'ip_allocator.exe', 'infrastructure_tool.exe']:
                        term_result = crowdstrike.terminate_process(device['device_id'], proc['process_id'])
                        if term_result.get('status') == 'terminated':
                            terminated_processes.append(f"{device.get('hostname', 'unknown')}:{proc['process_name']}")
                            print(f"   Terminated process: {proc['process_name']} on {device.get('hostname', 'unknown')}")
    except Exception as e:
        print(f"   Process termination failed: {e}")

# Remove acquired infrastructure resources
print(f"\n[ACTION] Removing acquired infrastructure resources...")
try:
    # Remove cloud resources
    for system in affected_systems:
        if system.get('type') == 'cloud_provisioning' and system.get('resource_name'):
            if 'aws' in system.get('service', '').lower():
                remove_result = shuffle.remove_aws_resource(system['resource_name'])
                if remove_result.get('status') == 'removed':
                    removed_resources.append(f"AWS: {system['resource_name']}")
                    print(f"   Removed AWS resource: {system['resource_name']}")
            elif 'azure' in system.get('service', '').lower():
                remove_result = shuffle.remove_azure_resource(system['resource_name'])
                if remove_result.get('status') == 'removed':
                    removed_resources.append(f"Azure: {system['resource_name']}")
                    print(f"   Removed Azure resource: {system['resource_name']}")
            elif 'gcp' in system.get('service', '').lower():
                remove_result = shuffle.remove_gcp_resource(system['resource_name'])
                if remove_result.get('status') == 'removed':
                    removed_resources.append(f"GCP: {system['resource_name']}")
                    print(f"   Removed GCP resource: {system['resource_name']}")

    # Cancel domain registrations
    for system in affected_systems:
        if system.get('type') == 'domain_registration' and system.get('domain'):
            cancel_result = shuffle.cancel_domain_registration(system['domain'])
            if cancel_result.get('status') == 'cancelled':
                removed_resources.append(f"Domain: {system['domain']}")
                print(f"   Cancelled domain registration: {system['domain']}")

    # Release IP allocations
    for system in affected_systems:
        if system.get('type') == 'ip_acquisition' and system.get('ip_range'):
            release_result = shuffle.release_ip_allocation(system['ip_range'])
            if release_result.get('status') == 'released':
                removed_resources.append(f"IP Range: {system['ip_range']}")
                print(f"   Released IP allocation: {system['ip_range']}")
except Exception as e:
    print(f"   Resource removal failed: {e}")

# Clean up artifacts and logs
print(f"\n[ACTION] Cleaning up artifacts and logs...")
try:
    # Clean infrastructure acquisition logs
    log_cleanup = splunk.clean_infrastructure_logs()
    if log_cleanup.get('status') == 'cleaned':
        cleaned_artifacts.append(f"Logs cleaned: {log_cleanup.get('count', 0)} entries")
        print(f"   Cleaned up {log_cleanup.get('count', 0)} infrastructure acquisition log entries")

    # Remove temporary files
    for system in affected_systems:
        if system.get('src_ip'):
            devices = crowdstrike.get_devices_by_ip(system['src_ip'])
            for device in devices:
                files = crowdstrike.get_infrastructure_files(device['device_id'])
                for file_path in files:
                    remove_result = crowdstrike.remove_file(device['device_id'], file_path)
                    if remove_result.get('status') == 'removed':
                        cleaned_artifacts.append(f"File: {device.get('hostname', 'unknown')}:{file_path}")
                        print(f"   Removed file: {file_path} from {device.get('hostname', 'unknown')}")
except Exception as e:
    print(f"   Artifact cleanup failed: {e}")

# Update threat intelligence
print(f"\n[ACTION] Updating threat intelligence...")
try:
    eradication_iocs = {
        'infrastructure_domains': [s.get('domain') for s in affected_systems if s.get('domain')],
        'infrastructure_ips': [s.get('ip_range') for s in affected_systems if s.get('ip_range')],
        'compromised_accounts': [s.get('user') for s in affected_systems if s.get('user')],
        'cloud_resources': [s.get('resource_name') for s in affected_systems if s.get('resource_name')]
    }
    misp.share_iocs(eradication_iocs, 'infrastructure_acquisition_eradicated')
    print(f"   Shared eradication IOCs with MISP threat intelligence")
except Exception as e:
    print(f"   Threat intelligence update failed: {e}")

# Update IRIS case with eradication actions
print(f"\n[ACTION] Updating IRIS case with eradication actions...")
try:
    eradication_summary = {
        'terminated_processes': len(terminated_processes),
        'removed_resources': len(removed_resources),
        'cleaned_artifacts': len(cleaned_artifacts),
        'eradication_time': eradication_time
    }
    iris.update_case(incident_id, 'eradication_complete', eradication_summary)
    print(f"   Updated IRIS case {incident_id} with eradication results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(terminated_processes)} malicious processes terminated")
print(f"  - {len(removed_resources)} infrastructure resources removed")
print(f"  - {len(cleaned_artifacts)} artifacts cleaned up")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
restored_systems = []
recreated_accounts = []
restored_services = []

# Restore isolated systems
print(f"\n[ACTION] Restoring isolated systems...")
for system in isolated_systems:
    try:
        # Find device by hostname/IP and restore
        if system in [s.get('src_ip') for s in affected_systems]:
            devices = crowdstrike.get_devices_by_ip(system)
            for device in devices:
                restore_result = crowdstrike.restore_host(device['device_id'])
                if restore_result.get('status') == 'restored':
                    restored_systems.append(device.get('hostname', system))
                    print(f"   Restored system: {device.get('hostname', system)}")
    except Exception as e:
        print(f"   System restoration failed for {system}: {e}")

# Recreate disabled accounts
print(f"\n[ACTION] Recreating disabled accounts...")
for user in disabled_accounts:
    try:
        recreate_result = shuffle.recreate_user_account(user)
        if recreate_result.get('status') == 'recreated':
            recreated_accounts.append(user)
            print(f"   Recreated account: {user}")
    except Exception as e:
        print(f"   Account recreation failed for {user}: {e}")

# Restore affected services
print(f"\n[ACTION] Restoring affected services...")
try:
    # Restore cloud management services
    cloud_restore = shuffle.restore_cloud_management()
    if cloud_restore.get('status') == 'restored':
        restored_services.append('Cloud management service')
        print(f"   Restored cloud management service")

    # Restore DNS services
    dns_restore = shuffle.restore_dns_service()
    if dns_restore.get('status') == 'restored':
        restored_services.append('DNS service')
        print(f"   Restored DNS service")

    # Restore network monitoring
    network_restore = shuffle.restore_network_monitoring()
    if network_restore.get('status') == 'restored':
        restored_services.append('Network monitoring')
        print(f"   Restored network monitoring service")
except Exception as e:
    print(f"   Service restoration failed: {e}")

# Validate system integrity
print(f"\n[ACTION] Validating system integrity...")
integrity_checks = []
try:
    for system in affected_systems:
        if system.get('src_ip'):
            devices = crowdstrike.get_devices_by_ip(system['src_ip'])
            for device in devices:
                integrity_result = crowdstrike.validate_system_integrity(device['device_id'])
                if integrity_result.get('status') == 'valid':
                    integrity_checks.append(device.get('hostname', system['src_ip']))
                    print(f"   System integrity validated: {device.get('hostname', system['src_ip'])}")
                else:
                    print(f"   System integrity issues detected: {device.get('hostname', system['src_ip'])}")
except Exception as e:
    print(f"   Integrity validation failed: {e}")

# Re-enable standard monitoring
print(f"\n[ACTION] Re-enabling standard monitoring...")
try:
    # Restore normal CrowdStrike monitoring
    cs_restore = crowdstrike.restore_standard_monitoring()
    if cs_restore.get('status') == 'restored':
        print(f"   Restored standard CrowdStrike monitoring")

    # Restore normal Splunk correlation rules
    splunk_restore = splunk.restore_standard_correlation()
    if splunk_restore.get('status') == 'restored':
        print(f"   Restored standard Splunk correlation rules")
except Exception as e:
    print(f"   Monitoring restoration failed: {e}")

# Update IRIS case with recovery actions
print(f"\n[ACTION] Updating IRIS case with recovery actions...")
try:
    recovery_summary = {
        'restored_systems': len(restored_systems),
        'recreated_accounts': len(recreated_accounts),
        'restored_services': len(restored_services),
        'integrity_validated': len(integrity_checks),
        'recovery_time': recovery_time
    }
    iris.update_case(incident_id, 'recovery_complete', recovery_summary)
    print(f"   Updated IRIS case {incident_id} with recovery results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored")
print(f"  - {len(recreated_accounts)} accounts recreated")
print(f"  - {len(restored_services)} services restored")
print(f"  - {len(integrity_checks)} systems validated")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

post_incident_time = datetime.now().isoformat()

# Generate incident report
print(f"\n[ACTION] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'technique': 'Acquire Infrastructure (T1583)',
        'detection_time': detection_time,
        'containment_time': containment_time,
        'eradication_time': eradication_time,
        'recovery_time': recovery_time,
        'post_incident_time': post_incident_time,
        'affected_systems': len(affected_systems),
        'isolated_systems': len(isolated_systems),
        'blocked_domains': len(blocked_domains),
        'blocked_ips': len(blocked_ips),
        'disabled_accounts': len(disabled_accounts),
        'terminated_processes': len(terminated_processes),
        'removed_resources': len(removed_resources),
        'cleaned_artifacts': len(cleaned_artifacts),
        'restored_systems': len(restored_systems),
        'recreated_accounts': len(recreated_accounts),
        'restored_services': len(restored_services),
        'integrity_validated': len(integrity_checks),
        'threat_intelligence_shared': True,
        'lessons_learned': [
            'Enhanced monitoring for bulk domain registrations needed',
            'Cloud resource provisioning alerts require tuning',
            'IP allocation monitoring should be improved',
            'Cross-platform infrastructure acquisition detection needed'
        ]
    }

    # Save report to IRIS
    iris.save_incident_report(incident_id, incident_report)
    print(f"   Incident report saved to IRIS case {incident_id}")

    # Share anonymized report with MISP
    misp.share_incident_report(incident_report, 'infrastructure_acquisition_response')
    print(f"   Anonymized incident report shared with MISP community")
except Exception as e:
    print(f"   Report generation failed: {e}")

# Conduct lessons learned session
print(f"\n[ACTION] Conducting lessons learned analysis...")
try:
    lessons_learned = {
        'what_went_well': [
            'Rapid detection of suspicious domain registrations',
            'Effective containment through resource blocking',
            'Comprehensive cloud resource cleanup',
            'Good threat intelligence integration'
        ],
        'what_could_improve': [
            'Earlier detection of infrastructure acquisition patterns',
            'Better correlation between different acquisition methods',
            'Enhanced monitoring for anonymous hosting providers',
            'Automated resource cleanup workflows'
        ],
        'preventive_measures': [
            'Implement domain registration monitoring and alerting',
            'Set up cloud resource provisioning limits and alerts',
            'Monitor IP allocation patterns and anomalies',
            'Regular review of infrastructure acquisition policies',
            'Enhanced threat hunting for infrastructure acquisition campaigns'
        ]
    }

    iris.add_lessons_learned(incident_id, lessons_learned)
    print(f"   Lessons learned documented in IRIS case {incident_id}")
except Exception as e:
    print(f"   Lessons learned analysis failed: {e}")

# Update security controls
print(f"\n[ACTION] Updating security controls...")
try:
    # Update Splunk correlation rules
    splunk.update_infrastructure_rules(incident_report)
    print(f"   Updated Splunk infrastructure acquisition rules")

    # Update CrowdStrike policies
    crowdstrike.update_infrastructure_policies(incident_report)
    print(f"   Updated CrowdStrike infrastructure monitoring policies")

    # Update Shuffle workflows
    shuffle.update_infrastructure_workflows(incident_report)
    print(f"   Updated Shuffle infrastructure response workflows")
except Exception as e:
    print(f"   Security controls update failed: {e}")

# Close incident
print(f"\n[ACTION] Closing incident...")
try:
    closure_summary = {
        'closure_time': post_incident_time,
        'incident_status': 'resolved',
        'follow_up_actions': [
            'Monitor for similar infrastructure acquisition campaigns',
            'Review cloud resource provisioning policies',
            'Conduct infrastructure acquisition threat hunting',
            'Update domain registration monitoring'
        ],
        'metrics': {
            'time_to_detect': 'Calculated from detection_time',
            'time_to_contain': 'Calculated from containment_time',
            'time_to_eradicate': 'Calculated from eradication_time',
            'time_to_recover': 'Calculated from recovery_time',
            'affected_assets': len(affected_systems)
        }
    }

    iris.close_case(incident_id, closure_summary)
    print(f"   IRIS case {incident_id} closed successfully")
except Exception as e:
    print(f"   Incident closure failed: {e}")

print(f"\n Post-incident activities complete:")
print(f"  - Incident report generated and shared")
print(f"  - Lessons learned documented")
print(f"  - Security controls updated")
print(f"  - Incident case closed")

print(f"\n Infrastructure acquisition incident response completed successfully!")
print(f"   Incident ID: {incident_id}")
print(f"   Total duration: {len(affected_systems)} infrastructure indicators mitigated")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
